# Connector: secrets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_secrets.ipynb)

Config values like `${JIRA_TOKEN}` are resolved at use time through a `Secrets` connector: environment variables, **HashiCorp Vault**, or **AWS Secrets Manager**. Resolved values are never written to `.pecca/config.resolved.yaml`. Secret values are masked in this notebook.
> Needs **Docker** for the container cells; without it (`PECCA_SKIP_DOCKER=1`) those cells are skipped and a fallback runs.

In [1]:
%pip install -q pecca hvac boto3 moto pyyaml requests

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
import yaml
from pecca.config.apply import apply as apply_config

mask = lambda v: v[:2] + "*" * (len(v) - 2) if v else v
def write_config(secrets_cfg):
    Path("pecca.yaml").write_text(yaml.safe_dump({"version": 1, "workspace": {"name": "default", "secrets": secrets_cfg}, "projects": {"default": {
        "governance": {"extends": "none"},
        "integrations": {"notifier": {"type": "slack", "webhook": "${SLACK_WEBHOOK}"}}}}}))
    context.reset()

## 1. Environment variables (default)

In [4]:
os.environ["SLACK_WEBHOOK"] = "https://hooks.example/env-secret"
write_config({"provider": "env"})
diff, resolved = apply_config("pecca.yaml")
stored = Path(os.environ["PECCA_HOME"], "config.resolved.yaml").read_text()
print("placeholder kept on disk:", "${SLACK_WEBHOOK}" in stored, "| value on disk:", "env-secret" in stored)
del os.environ["SLACK_WEBHOOK"]
try:
    apply_config("pecca.yaml")
except pecca.core.errors.PeccaError as e:
    print("unresolved ->", e.message)

placeholder kept on disk: True | value on disk: False
unresolved -> unresolved variables: SLACK_WEBHOOK


## 2. HashiCorp Vault (dev server in Docker)
A throwaway `hashicorp/vault` container in dev mode with a random root token, bound to localhost.

In [5]:
import secrets as _secrets
vault = {"name": f"pecca-vault-{_secrets.token_hex(3)}", "token": _secrets.token_hex(12), "port": free_port()}
if USE_DOCKER:
    import requests
    cleanup_later("docker", "rm", "-f", vault["name"])
    sh("docker", "run", "-d", "--rm", "--cap-add=IPC_LOCK", "--name", vault["name"], "-p", f"127.0.0.1:{vault['port']}:8200",
       "-e", f"VAULT_DEV_ROOT_TOKEN_ID={vault['token']}", "-e", "VAULT_DEV_LISTEN_ADDRESS=0.0.0.0:8200", "hashicorp/vault:1.18")
    base = f"http://127.0.0.1:{vault['port']}"
    wait_for(lambda: requests.get(f"{base}/v1/sys/health", timeout=2).status_code == 200, "Vault")
    requests.post(f"{base}/v1/secret/data/pecca", headers={"X-Vault-Token": vault["token"]},
                  json={"data": {"SLACK_WEBHOOK": "https://hooks.example/vault-secret"}}).raise_for_status()
    write_config({"provider": "vault", "url": base, "token": vault["token"], "path": "pecca", "mount": "secret"})
    diff, resolved = apply_config("pecca.yaml")
    value = context.get_workspace().secrets.get("SLACK_WEBHOOK")
    print("resolved from Vault:", mask(value))
    print("value on disk:", "vault-secret" in Path(os.environ["PECCA_HOME"], "config.resolved.yaml").read_text())
    sh("docker", "rm", "-f", vault["name"], check=False)
    print("Vault container removed")
else:
    print("skipped: needs Docker (the connector call is `secrets: {provider: vault, url, token, path, mount}`).")

resolved from Vault: ht********************************
value on disk: False
Vault container removed


## 3. AWS Secrets Manager (mocked with moto)

In [6]:
import boto3
from moto import mock_aws

for k, v in {"AWS_ACCESS_KEY_ID": "testing", "AWS_SECRET_ACCESS_KEY": "testing", "AWS_DEFAULT_REGION": "eu-west-1"}.items():
    os.environ[k] = v
with mock_aws():
    boto3.client("secretsmanager").create_secret(Name="pecca/SLACK_WEBHOOK", SecretString="https://hooks.example/aws-secret")
    write_config({"provider": "aws-secrets-manager", "region": "eu-west-1", "prefix": "pecca/"})
    apply_config("pecca.yaml")
    print("resolved from AWS Secrets Manager (mock):", mask(context.get_workspace().secrets.get("SLACK_WEBHOOK")))

resolved from AWS Secrets Manager (mock): ht******************************
